In [1]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    if "images" in root:
        continue
    for f in files:
        if f.endswith(".parquet") or f.endswith(".csv"):
            print(os.path.join(root, f))

/kaggle/input/notebooks/fatimawaseem1111/parquet/test.parquet
/kaggle/input/notebooks/fatimawaseem1111/parquet/train.parquet
/kaggle/input/competitions/h-and-m-personalized-fashion-recommendations/sample_submission.csv
/kaggle/input/competitions/h-and-m-personalized-fashion-recommendations/articles.csv
/kaggle/input/competitions/h-and-m-personalized-fashion-recommendations/transactions_train.csv
/kaggle/input/competitions/h-and-m-personalized-fashion-recommendations/customers.csv


In [2]:
import pandas as pd
train = pd.read_parquet("/kaggle/input/notebooks/fatimawaseem1111/parquet/train.parquet")
test = pd.read_parquet("/kaggle/input/notebooks/fatimawaseem1111/parquet/test.parquet")
print(train.shape, test.shape)

(534626, 5) (125341, 5)


In [3]:
ground_truth = test.groupby("customer_id")["article_id"].apply(set).to_dict()
print(len(ground_truth))

33168


In [4]:
def apk(actual, predicted, k=12):
    if not actual:
        return 0.0
    score = 0.0
    hits = 0
    for i, p in enumerate(predicted[:k]):
        if p in actual and p not in predicted[:i]:
            hits += 1
            score += hits / (i + 1)
    return score / min(len(actual), k)

In [5]:
print(apk({"A", "B"}, ["X", "A", "Y", "B"]))
print(apk({"A", "B"}, ["A", "B", "X", "Y"]))

0.5
1.0


In [6]:
def recall_at_k(actual, predicted, k=12):
    if not actual:
        return 0.0
    hits = len(set(predicted[:k]) & actual)
    return hits / len(actual)

In [7]:
print(recall_at_k({"A", "B"}, ["X", "A", "Y", "B"]))   # should be 1.0 (both found)
print(recall_at_k({"A", "B"}, ["X", "A", "Y", "Z"]))   # should be 0.5 (one found)

1.0
0.5


In [8]:
import numpy as np

def evaluate(recs, ground_truth, k=12):
    maps, recalls = [], []
    for cust, actual in ground_truth.items():
        predicted = recs.get(cust, [])
        maps.append(apk(actual, predicted, k))
        recalls.append(recall_at_k(actual, predicted, k))
    return {"MAP@12": np.mean(maps), "Recall@12": np.mean(recalls)}

In [9]:
ground_truth = test.groupby("customer_id")["article_id"].apply(set).to_dict()

In [10]:
last_week = train[train["t_dat"] >= train["t_dat"].max() - pd.Timedelta(days=6)]
top12 = last_week["article_id"].value_counts().index[:12].tolist()

recs_popular = {cust: top12 for cust in ground_truth}
print(evaluate(recs_popular, ground_truth))

{'MAP@12': np.float64(0.010143660851050345), 'Recall@12': np.float64(0.027376086485596005)}


In [11]:
rng = np.random.default_rng(42)
all_items = train["article_id"].unique()

recs_random = {cust: rng.choice(all_items, 12, replace=False).tolist() for cust in ground_truth}
print(evaluate(recs_random, ground_truth))

{'MAP@12': np.float64(0.00012515248431643752), 'Recall@12': np.float64(0.0004676969413311671)}


In [12]:
recent = (train.sort_values("t_dat", ascending=False)
               .groupby("customer_id")["article_id"]
               .apply(lambda x: list(dict.fromkeys(x))[:12]))

recs_repurchase = {}
for cust in ground_truth:
    items = recent.get(cust, [])
    items = items + [i for i in top12 if i not in items]
    recs_repurchase[cust] = items[:12]

print(evaluate(recs_repurchase, ground_truth))

{'MAP@12': np.float64(0.03301332691433908), 'Recall@12': np.float64(0.06569949543865923)}


In [13]:
results = pd.DataFrame({
    "Random": evaluate(recs_random, ground_truth),
    "Most popular": evaluate(recs_popular, ground_truth),
    "Repurchase": evaluate(recs_repurchase, ground_truth),
}).T
print(results)
results.to_csv("/kaggle/working/baseline_results.csv")


                MAP@12  Recall@12
Random        0.000125   0.000468
Most popular  0.010144   0.027376
Repurchase    0.033013   0.065699


In [14]:
!pip install cornac

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 59.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [cornac]


In [15]:
from cornac.data import Dataset

uir = train[["customer_id", "article_id"]].drop_duplicates()
uir["rating"] = 1.0
train_set = Dataset.from_uir(uir.itertuples(index=False, name=None), seed=42)
print(train_set.num_users, "users |", train_set.num_items, "items")

33168 users | 28593 items


In [16]:
from cornac.data import Dataset
from cornac.models import BPR

uir = train[["customer_id", "article_id"]].drop_duplicates()
uir["rating"] = 1.0
train_set = Dataset.from_uir(uir.itertuples(index=False, name=None), seed=42)
print(train_set.num_users, "users |", train_set.num_items, "items")

bpr = BPR(k=64, max_iter=100, learning_rate=0.01, lambda_reg=0.01, seed=42, verbose=True)
bpr.fit(train_set)

print("Trained:", bpr.u_factors is not None)
print("User factors shape:", None if bpr.u_factors is None else bpr.u_factors.shape)

33168 users | 28593 items


  0%|          | 0/100 [00:00<?, ?it/s]

Optimization finished!
Trained: True
User factors shape: (33168, 64)


In [17]:
item_ids = np.array(list(train_set.iid_map.keys()))

recs_bpr = {}
for cust in ground_truth:
    uidx = train_set.uid_map[cust]
    scores = bpr.score(uidx)
    top = np.argsort(-scores)[:12]
    recs_bpr[cust] = item_ids[top].tolist()

print(evaluate(recs_bpr, ground_truth))

{'MAP@12': np.float64(0.003905982274729409), 'Recall@12': np.float64(0.013255651875945906)}


In [18]:
recs_hybrid = {}
for cust in ground_truth:
    items = recent.get(cust, [])
    items = items + [i for i in recs_bpr[cust] if i not in items]
    recs_hybrid[cust] = items[:12]

print(evaluate(recs_hybrid, ground_truth))

{'MAP@12': np.float64(0.03239153168173471), 'Recall@12': np.float64(0.06076188227050775)}


In [19]:
val_start = pd.Timestamp("2020-09-09")
train_inner = train[train["t_dat"] < val_start]
val = train[train["t_dat"] >= val_start]

val_truth = (val[val["customer_id"].isin(train_inner["customer_id"].unique())]
             .groupby("customer_id")["article_id"].apply(set).to_dict())
print(len(train_inner), "train rows |", len(val_truth), "validation customers")

488220 train rows | 9724 validation customers


In [20]:
import time
def train_and_score(data, truth, k, lr, reg, iters=100):
    u = data[["customer_id", "article_id"]].drop_duplicates()
    u["rating"] = 1.0
    ds = Dataset.from_uir(u.itertuples(index=False, name=None), seed=42)
    model = BPR(k=k, max_iter=iters, learning_rate=lr, lambda_reg=reg, seed=42, verbose=False)
    model.fit(ds)
    ids = np.array(list(ds.iid_map.keys()))
    recs = {c: ids[np.argsort(-model.score(ds.uid_map[c]))[:12]].tolist() for c in truth}
    return evaluate(recs, truth)["MAP@12"]

In [21]:
import time 
results_tuning = []
for k in [32, 64, 128]:
    for lr in [0.01, 0.05]:
        for reg in [0.001, 0.01]:
            score = train_and_score(train_inner, val_truth, k, lr, reg)
            results_tuning.append((k, lr, reg, score))
            print(f"k={k}, lr={lr}, reg={reg} -> val MAP@12={score:.4f}", flush=True)

k=32, lr=0.01, reg=0.001 -> val MAP@12=0.0036
k=32, lr=0.01, reg=0.01 -> val MAP@12=0.0039
k=32, lr=0.05, reg=0.001 -> val MAP@12=0.0041
k=32, lr=0.05, reg=0.01 -> val MAP@12=0.0048
k=64, lr=0.01, reg=0.001 -> val MAP@12=0.0044
k=64, lr=0.01, reg=0.01 -> val MAP@12=0.0044
k=64, lr=0.05, reg=0.001 -> val MAP@12=0.0067
k=64, lr=0.05, reg=0.01 -> val MAP@12=0.0067
k=128, lr=0.01, reg=0.001 -> val MAP@12=0.0049
k=128, lr=0.01, reg=0.01 -> val MAP@12=0.0044
k=128, lr=0.05, reg=0.001 -> val MAP@12=0.0082
k=128, lr=0.05, reg=0.01 -> val MAP@12=0.0077


In [22]:
tuning_df = pd.DataFrame(results_tuning, columns=["k", "lr", "reg", "val_MAP@12"])
tuning_df = tuning_df.sort_values("val_MAP@12", ascending=False)
print(tuning_df)
tuning_df.to_csv("/kaggle/working/bpr_tuning.csv", index=False)

best = tuning_df.iloc[0]
print("Best:", best.to_dict())

      k    lr    reg  val_MAP@12
10  128  0.05  0.001    0.008248
11  128  0.05  0.010    0.007742
7    64  0.05  0.010    0.006698
6    64  0.05  0.001    0.006687
8   128  0.01  0.001    0.004897
3    32  0.05  0.010    0.004812
9   128  0.01  0.010    0.004387
4    64  0.01  0.001    0.004363
5    64  0.01  0.010    0.004352
2    32  0.05  0.001    0.004054
1    32  0.01  0.010    0.003917
0    32  0.01  0.001    0.003564
Best: {'k': 128.0, 'lr': 0.05, 'reg': 0.001, 'val_MAP@12': 0.008247918384129376}


In [23]:
bpr_best = BPR(k=int(best["k"]), max_iter=100, learning_rate=best["lr"],
               lambda_reg=best["reg"], seed=42, verbose=True)
bpr_best.fit(train_set)

recs_bpr_tuned = {}
for cust in ground_truth:
    scores = bpr_best.score(train_set.uid_map[cust])
    recs_bpr_tuned[cust] = item_ids[np.argsort(-scores)[:12]].tolist()

print(evaluate(recs_bpr_tuned, ground_truth))

  0%|          | 0/100 [00:00<?, ?it/s]

Optimization finished!
{'MAP@12': np.float64(0.009195470799949144), 'Recall@12': np.float64(0.02821977099823048)}


In [24]:
results.loc["BPR (default)"] = evaluate(recs_bpr, ground_truth)
results.loc["BPR (tuned)"] = evaluate(recs_bpr_tuned, ground_truth)
print(results)
results.to_csv("/kaggle/working/baseline_results.csv")

                 MAP@12  Recall@12
Random         0.000125   0.000468
Most popular   0.010144   0.027376
Repurchase     0.033013   0.065699
BPR (default)  0.003906   0.013256
BPR (tuned)    0.009195   0.028220
